# RAG From Scratch — Query Transformations 

**Query transformations** are a family of techniques that rewrite, expand, or decompose the user's question *before* it hits the retriever, instead of embedding the raw question as-is. The motivation: a single question, embedded once, is a narrow probe into vector space — phrasing, ambiguity, and multi-part questions all limit what a single similarity search can find. Each technique below attacks that limitation differently.

| Part | Technique | Idea in one line |
|---|---|---|
| 5 | Multi-Query | Ask the LLM to rephrase the question N different ways, retrieve for each, union the results |
| 6 | RAG-Fusion | Same idea as Multi-Query, but merge results with **reciprocal rank fusion** instead of a plain union |
| 7 | Decomposition | Break one question into sub-questions, answer them (recursively or independently), then synthesize |
| 8 | Step-Back | Ask a more *abstract* version of the question first, retrieve broader context, then answer the specific one |
| 9 | HyDE | Have the LLM **hallucinate** a plausible answer first, embed *that* instead of the question, and retrieve with it |



#### What changes vs. the OpenAI version, and what doesn't

| Role | OpenAI version | This notebook |
|---|---|---|
| Embedding model | `OpenAIEmbeddings` (API call, costs money) | `HuggingFaceEmbeddings` — local, free |
| Chat / generation model | `ChatOpenAI(temperature=0)` | `HuggingFacePipeline` + `ChatHuggingFace` — local, free |
| API keys | Required (`OPENAI_API_KEY`) | None required |
| Query-transformation logic itself | — | **Identical.** Multi-query fan-out, RRF, decomposition, step-back, HyDE are all just prompt engineering + `Runnable` composition — they don't know or care which LLM is answering. |



In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = 0 if torch.cuda.is_available() else -1  # transformers convention: GPU index, or -1 for CPU

CUDA available: True
GPU: NVIDIA GeForce RTX 4080 SUPER


### Models used in this notebook

Same two models as the indexing notebook, for a fair apples-to-apples comparison of technique, not model size:

- **`sentence-transformers/all-MiniLM-L6-v2`** for embeddings
- **`Qwen/Qwen2.5-1.5B-Instruct`** for generation — small enough to run fast even though this notebook makes noticeably *more* LLM calls per question than the basic pipeline (multi-query alone fires 5 retrievals off of one generation call; decomposition and step-back each make several sequential calls)

In [2]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

### Optional: LangSmith tracing

This notebook is a *great* candidate for tracing — multi-query fans one question out into 5 retrievals, decomposition chains several LLM calls in sequence, and it's easy to lose track of what actually got retrieved for which sub-question. If you've already run [0_langsmith_setup.ipynb](0_langsmith_setup.ipynb) and have a `.env` with `LANGSMITH_API_KEY` set, the cell below turns tracing on for this notebook and gives it its own project name so its runs don't mix with the indexing notebook's. If you haven't set that up, this cell just skips tracing — everything else below still works identically, you just won't have a UI to inspect the chains in.

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get("LANGSMITH_TRACING") == "true" and os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_PROJECT"] = "rag-from-scratch-query-transformations"
    print(f"LangSmith tracing ON -> project '{os.environ['LANGSMITH_PROJECT']}'")
else:
    print("LangSmith tracing OFF (no .env / LANGSMITH_API_KEY found) - continuing without it.")
    print("See 0_langsmith_setup.ipynb to enable it.")

LangSmith tracing ON -> project 'rag-from-scratch-query-transformations'


### Build the LLM and embedding model once, reuse everywhere

This is worth calling out explicitly because it's a real difference from the OpenAI notebook, not just a cosmetic one. `ChatOpenAI(temperature=0)` is a thin, cheap HTTP client — the original notebook casually re-instantiates it in almost every cell. `HuggingFacePipeline.from_model_id(...)` is not cheap: it loads real weights onto the GPU/CPU. Re-running it in every cell would reload the model from scratch each time.

So instead of recreating the LLM per section, we build `llm` and `embeddings` **once**, here, and every technique below (Multi-Query, RAG-Fusion, Decomposition, Step-Back, HyDE) reuses the same two objects. This is generally the right pattern for local models regardless of tutorial context.

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id=LLM_MODEL,
    task="text-generation",
    device=DEVICE,
    pipeline_kwargs={
        "max_new_tokens": 512,
        "do_sample": False,        # deterministic, same intent as ChatOpenAI(temperature=0)
        "return_full_text": False,
    },
)
llm = ChatHuggingFace(llm=llm_pipeline)

c:\Users\Amine\PycharmProjects\rag-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 4443.06it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Indexing

Same source article as the other notebooks (Lilian Weng's post on LLM-powered autonomous agents), so retrieval quality is directly comparable across notebooks. Nothing here is new if you've been through [1_indexing_retrieval_generation.ipynb](1_indexing_retrieval_generation.ipynb) — load, split, embed, store — just condensed into one cell since indexing itself isn't the focus of this notebook.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(blog_docs)


vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print(f"Indexed {len(splits)} chunks")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Indexed 214 chunks


---
## Part 5: Multi-Query

**Idea:** distance-based similarity search is brittle to exact phrasing — a question worded slightly differently can land in a different part of embedding space and miss relevant chunks. Multi-Query asks the LLM to generate several *different phrasings* of the same question, retrieves separately for each, then takes the union of all retrieved chunks. More phrasings → more chances one of them lands close to the relevant chunks in vector space.

Flow: `question → LLM generates 5 rephrasings → retrieve for each → union of unique docs → answer`

Docs: https://reference.langchain.com/python/langchain-classic/retrievers/multi_query/MultiQueryRetriever

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Create the generate queries Runnable, which takes a question string and returns a list of alternative queries
# it does this by:
#    a) filling the prompt with the question
#    b) sending the filled prompt to the model, getting the LLM response back
#    c) parse the LLM response into a list of strings (Each string is a different perspective on the original question) 
generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)

generate_queries.invoke("What are the subjects of the field of Physics?")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


['1. Which topics or areas does the field of Physics cover?',
 '2. What main branches or fields can be identified within the discipline of Physics?',
 '3. In what specific domains or subfields does Physics focus its research and study?',
 '4. What are the primary subject matters explored in the domain of Physics?',
 '5. How many distinct categories or themes do we find when examining the subjects of Physics?']

**A note on smaller local models:** GPT-3.5 reliably outputs exactly 5 clean lines here. A 1.5B model is less consistent — it might add numbering (`1. ...`), a preamble sentence, or merge two questions onto one line. The `if q.strip()` filter above just drops blank lines; if you print `generate_queries.invoke(...)` and see messier output, that's expected local-model behavior, not a bug — feel free to tighten the prompt or post-process further.

In [7]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs across all rephrasings"""
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]

question = "What is task decomposition for LLM agents?"

# generate_queries.invoke(question) returns a list of rephrased versions of the question
# retriever.map() runs the retrieval for each question, each question gets its own list of retrieved docs, 
# the result is a list of lists of doc
# get_unique_union() takes the list of lists of docs and returns a single list of unique docs
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Unique docs retrieved across all rephrasings: {len(docs)}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unique docs retrieved across all rephrasings: 8


C:\Users\Amine\AppData\Local\Temp\ipykernel_27432\584704326.py:7: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
C:\Users\Amine\AppData\Local\Temp\ipykernel_27432\584704326.py:7: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [8]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
question = "What is task decomposition for LLM agents?"


template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
C:\Users\Amine\AppData\Local\Temp\ipykernel_27432\584704326.py:7: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Task decomposition for LLM agents refers to breaking down complex tasks into smaller, more manageable subtasks that can be handled individually. This process helps in organizing the work efficiently and allows the agent to focus on specific parts of the larger task at hand. By decomposing tasks, LLM agents can better handle intricate and multifaceted problems, improving their overall performance and adaptability.'

---
## Part 6: RAG-Fusion

**Idea:** same fan-out as Multi-Query (LLM generates several related search queries), but instead of **a flat union** between each list of documents for each query, results are merged with **Reciprocal Rank Fusion (RRF)** — a document that ranks highly across *several* of the query variants scores higher than one that only shows up once. This rewards documents that are consistently relevant rather than a coincidental single hit.


$$\mathrm{RRF}(d)=\sum_{i=1}^{N}\frac{1}{k+\mathrm{rank}_i(d)}$$

Think of **RRF (Reciprocal Rank Fusion)** as a way to combine the rankings from several searches.


### What each part means

- **\(d\)** → the document/chunk we are scoring.
- **\(N\)** → number of different search results/rankings.
- **rank_i(d)** → the position of document \(d\) in ranking \(i\).
  - 1st place → rank = 1
  - 2nd place → rank = 2
  - etc.
- **\(k\)** → a constant that reduces the impact of very high rankings. Usually **60**.
- **\(\sum\)** → add the scores the document gets from all rankings.

### Simple example

Suppose RAG-Fusion generates **3 queries**, and a document appears at:

| Query | Document rank |
|---|---:|
| Query 1 | 1 |
| Query 2 | 3 |
| Query 3 | 10 |

With \(k=60\), the RFF score of that document would be : 

$$
\begin{aligned}
\mathrm{RRF}(d)
&= \frac{1}{60+1}
+ \frac{1}{60+3}
+ \frac{1}{60+10} \\
&= 0.0164 + 0.0159 + 0.0143 \\
&= \mathbf{0.0466}
\end{aligned}
$$
So the **higher the RRF score, the better the document performed across the different searches**.

The key idea is: **a document that consistently appears near the top of multiple rankings gets a high RRF score.**

Blog: https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1

In [9]:
# RAG-Fusion: Related queries
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)

In [10]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """Reciprocal Rank Fusion over multiple ranked lists of documents.

    Each doc's score is the sum of 1/(rank + k) across every list it appears in -
    consistently-ranked docs across multiple query variants score higher than a
    single high rank in only one list.
    """
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
print(f"Fused, reranked docs: {len(docs)}")
print(f"Top doc score: {docs[0][1]:.4f}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fused, reranked docs: 8
Top doc score: 0.0659


C:\Users\Amine\AppData\Local\Temp\ipykernel_27432\3664301716.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [11]:
final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
C:\Users\Amine\AppData\Local\Temp\ipykernel_27432\3664301716.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Task decomposition for LLM agents refers to breaking down complex tasks into smaller, more manageable subtasks that can be handled individually. This process helps in organizing the workload efficiently and enables better understanding and execution of larger tasks through simpler, step-by-step approaches.'

---
## Part 7: Decomposition

**Idea:** some questions are really several questions stacked together (e.g. "what are the main components of X and how do they interact?"). Decomposition asks the LLM to split the input into standalone sub-questions, then answers them one of two ways:

- **Recursively** — answer sub-question 1, feed that Q&A pair as context into sub-question 2, and so on, building up context as you go (good when sub-questions depend on each other)
- **Individually** — answer every sub-question independently in parallel, then synthesize all the Q&A pairs into one final answer (good when sub-questions are independent)

Papers: https://arxiv.org/pdf/2205.10625.pdf , https://arxiv.org/abs/2212.10509.pdf

In [ ]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)

question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question": question})
questions

### Answer recursively

Each sub-question is answered with retrieved context *plus* every previous sub-question's Q&A pair, so later answers can build on earlier ones.

In [ ]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""
decomposition_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
def format_qa_pair(question, answer):
    """Format one Q and A pair"""
    formatted_string = f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

q_a_pairs = ""
for q in questions:
    rag_chain = (
        {"context": itemgetter("question") | retriever,
         "question": itemgetter("question"),
         "q_a_pairs": itemgetter("q_a_pairs")}
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )

    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair

print(answer)  # the final sub-question's answer, informed by all the ones before it

### Answer individually

Same sub-questions, but each is answered independently (no shared context between them), then all Q&A pairs are synthesized into one final answer at the end. The original notebook pulls a community prompt from LangChain Hub (`hub.pull("rlm/rag-prompt")`); we inline the equivalent RAG prompt directly instead, to keep this notebook fully self-contained and avoid an extra network dependency.

In [ ]:
from langchain_core.runnables import RunnableLambda

# Standard RAG prompt (equivalent to the hub's "rlm/rag-prompt")
prompt_rag = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. Use the following "
    "pieces of retrieved context to answer the question. If you don't know "
    "the answer, just say that you don't know. Use three sentences maximum "
    "and keep the answer concise.\n"
    "Question: {question}\nContext: {context}\nAnswer:"
)

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """RAG on each sub-question, independently"""
    sub_questions = sub_question_generator_chain.invoke({"question": question})

    rag_results = []
    for sub_question in sub_questions:
        retrieved_docs = retriever.invoke(sub_question)
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)

    return rag_results, sub_questions

answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

In [ ]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""
synthesis_prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    synthesis_prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context": context, "question": question})

---
## Part 8: Step-Back

**Idea:** ask a more *abstract, generic* version of the question first (the "step-back" question), retrieve context for that broader question too, and answer the original specific question using **both** the specific and the broader context. Useful when the specific question needs background/world knowledge that a narrow retrieval alone wouldn't surface.

Paper: https://arxiv.org/pdf/2310.06117.pdf

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Few-shot examples showing the specific -> generic transformation
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel\u00e1\u0159's was born in what country?",
        "output": "what is Jan Sindel\u00e1\u0159's personal history?",
    },
]
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
step_back_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at world knowledge. Your task is to step back and "
            "paraphrase a question to a more generic step-back question, which is "
            "easier to answer. Here are a few examples:",
        ),
        few_shot_prompt,
        ("user", "{question}"),
    ]
)

In [ ]:
generate_queries_step_back = step_back_prompt | llm | StrOutputParser()

question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

In [ ]:
# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

step_back_chain = (
    {
        # Retrieve context using the normal (specific) question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back (generic) question
        "step_back_context": generate_queries_step_back | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

step_back_chain.invoke({"question": question})

---
## Part 9: HyDE (Hypothetical Document Embeddings)

**Idea:** instead of embedding the question directly, ask the LLM to write a *hypothetical, plausible-sounding answer* first — even though it's likely to contain hallucinated specifics — and embed **that** instead. The intuition: a fabricated answer is written in the same style and vocabulary as a real answer, so it can sit closer in embedding space to the actual relevant document than the original, differently-phrased question does.

Docs: https://github.com/langchain-ai/langchain/blob/master/cookbook/hypothetical_document_embeddings.ipynb

Paper: https://arxiv.org/abs/2212.10496

In [ ]:
# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser()
)

question = "What is task decomposition for LLM agents?"
hypothetical_doc = generate_docs_for_retrieval.invoke({"question": question})
hypothetical_doc

In [ ]:
# Retrieve using the hypothetical document instead of the raw question
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})
retrieved_docs

In [ ]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context": retrieved_docs, "question": question})

---
## Recap

| Technique | What it changes about retrieval | LLM calls per question |
|---|---|---|
| Multi-Query | Rephrase the question N ways, union retrieved docs | 1 (rephrasings) |
| RAG-Fusion | Same rephrasing, but merge with reciprocal rank fusion | 1 (rephrasings) |
| Decomposition | Split into sub-questions, answer recursively or independently | 1 (sub-questions) + 1 per sub-question |
| Step-Back | Retrieve for both the specific and a generalized version of the question | 1 (step-back question) |
| HyDE | Embed a hallucinated hypothetical answer instead of the question itself | 1 (hypothetical doc) |

### What changed vs. the OpenAI notebook

- `OpenAIEmbeddings()` -> `HuggingFaceEmbeddings(model_name=...)`
- `ChatOpenAI(temperature=0)` -> `HuggingFacePipeline.from_model_id(...)` wrapped in `ChatHuggingFace(llm=...)`, built **once** and reused across every part (see the note in Environment setup on why that matters for local models)
- `retriever.get_relevant_documents(...)` (deprecated) -> `retriever.invoke(...)`
- `hub.pull("rlm/rag-prompt")` -> an inlined equivalent prompt, to keep the notebook self-contained
- No API keys required anywhere

### What didn't change at all

Every technique's actual logic — the multi-query fan-out, `get_unique_union`, `reciprocal_rank_fusion`, the recursive/individual decomposition loops, the step-back few-shot prompt, the HyDE hypothetical-document trick — is byte-for-byte the same idea as the OpenAI notebook. That's the point being demonstrated across this whole repo: query transformation strategies are prompt-engineering and `Runnable` composition patterns that sit *on top of* whichever LLM you plug in underneath.

If you have [LangSmith tracing](0_langsmith_setup.ipynb) enabled, go compare the traces for these five techniques against each other and against the plain pipeline in notebook 1 — you'll see directly how much more retrieval and LLM work each of these does per question, which is exactly the cost/quality trade-off each technique is making.